# Semana 1 — Fundamentos: de la probabilidad condicional a los procesos estocásticos

**Curso:** Procesos Estocásticos · Ingeniería Industrial · Universidad EIA
**Bloque:** A — Procesos Estocásticos (Ross, cap. 1–3)
**Resultado de aprendizaje:** RA1 — *Modelar sistemas estocásticos discretos y continuos aplicando principios de probabilidad e ingeniería.*
**Tipo de notebook:** Contenido (no gestionado por nbgrader — todas las actividades de esta semana son formativas, sin nota).

---

**Objetivos de esta sesión** — cada actividad de este notebook está etiquetada con el objetivo que refuerza:

- **O1.** Repasar probabilidad condicional y valor esperado (incluida la ley de probabilidad/valor esperado totales) con el rigor que vas a necesitar el resto del curso.
- **O2.** Definir formalmente qué es un proceso estocástico y distinguirlo de una variable aleatoria simple.
- **O3.** Clasificar un proceso estocástico según su espacio de estados (discreto/continuo) y su parámetro temporal (discreto/continuo).
- **O4.** Simular en Python tu primer proceso estocástico simple (una caminata aleatoria) y relacionar sus parámetros con su comportamiento esperado.

**Conexión con lo que ya sabes:** en Simulación ya generaste variables aleatorias y ajustaste distribuciones. Esta semana damos el salto de *una* variable aleatoria a una *secuencia* de variables aleatorias relacionadas entre sí — eso es un proceso estocástico.

## 1. Repaso: probabilidad condicional y valor esperado condicional — *(O1)*

Antes de hablar de procesos, necesitamos tener muy afianzados dos conceptos que vas a usar constantemente: **probabilidad condicional** y **valor esperado condicional**.

Recuerda que la probabilidad condicional de un evento $A$ dado un evento $B$ es:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

Y el valor esperado condicional de una variable aleatoria $X$ dado un evento $B$:

$$E[X \mid B] = \sum_{x} x \cdot P(X = x \mid B)$$

De forma análoga, si $X$ es continua:

$$E[X \mid B] = \int_{-\infty}^{\infty} x \cdot f_X(x \mid B)\, dx$$

Ambas versiones (discreta y continua) las vas a necesitar constantemente — la discreta para cadenas de Markov (próximas 3 semanas), la continua desde la Semana 5 en adelante (proceso de Poisson, tiempos entre llegadas).

Esto es la base de **toda** la teoría de procesos estocásticos: cuando decimos que un proceso "recuerda" o "no recuerda" su pasado, estamos hablando exactamente de cómo cambian estas probabilidades condicionales.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

# Ejemplo: una máquina puede estar Operativa (0) o Dañada (1).
# P(Dañada mañana | Operativa hoy) = 0.05
# P(Dañada mañana | Dañada hoy)    = 0.70  (si no se repara)

P_dañada_dado_operativa = 0.05
P_dañada_dado_dañada = 0.70

print(f"P(Dañada mañana | Operativa hoy) = {P_dañada_dado_operativa}")
print(f"P(Dañada mañana | Dañada hoy)    = {P_dañada_dado_dañada}")
print()
print("Nota algo importante: estas dos probabilidades son DIFERENTES.")
print("El estado de HOY afecta la probabilidad de MAÑANA. Esa dependencia")
print("es exactamente lo que vamos a formalizar como 'proceso estocástico'.")

P(Dañada mañana | Operativa hoy) = 0.05
P(Dañada mañana | Dañada hoy)    = 0.7

Nota algo importante: estas dos probabilidades son DIFERENTES.
El estado de HOY afecta la probabilidad de MAÑANA. Esa dependencia
es exactamente lo que vamos a formalizar como 'proceso estocástico'.


Antes de resolver, recuerda la **ley de probabilidad total**: si $\{E_1, \ldots, E_n\}$ es una partición del espacio muestral (eventos mutuamente excluyentes que cubren todos los casos posibles), entonces para cualquier evento $A$:

$$P(A) = \sum_i P(A \mid E_i) \cdot P(E_i)$$

Y su versión análoga para valor esperado (**ley del valor esperado total**):

$$E[X] = \sum_i E[X \mid E_i] \cdot P(E_i)$$

En el ejercicio de abajo, la partición es simplemente $\{\text{Operativa hoy}, \text{Dañada hoy}\}$.

### Actividad 1 — Ley de probabilidad total y valor esperado total *(O1, formativa)*


En la práctica casi nunca conoces con certeza el estado de hoy — lo más común es tener una *creencia* (probabilidad) sobre él. Supón que, antes de revisar la máquina, tu estimado es:

$$P(\text{Operativa hoy}) = 0.90, \qquad P(\text{Dañada hoy}) = 0.10$$

Y que reparar la máquina si amanece dañada cuesta **\$5,000,000 COP**.

**Calcula, usando la celda de abajo:**

1. La probabilidad (marginal, no condicional) de que la máquina esté dañada *mañana* — usando la **ley de probabilidad total**: $P(\text{Dañada mañana}) = \sum_i P(\text{Dañada mañana} \mid E_i) \cdot P(E_i)$.
2. El costo esperado de reparación para mañana — usando la **ley de valor esperado total**, la misma idea pero aplicada a un valor esperado en vez de una probabilidad.

In [2]:
P_operativa_hoy = 0.90
P_dañada_hoy = 0.10
costo_reparacion = 5_000_000

# YOUR CODE HERE
P_dañada_mañana = None
E_costo_mañana = None

print("P(Dañada mañana) =", P_dañada_mañana)
print("E[costo mañana]  =", E_costo_mañana)

P(Dañada mañana) = None
E[costo mañana]  = None


<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

```python
P_dañada_mañana = P_operativa_hoy * P_dañada_dado_operativa + P_dañada_hoy * P_dañada_dado_dañada
# = 0.90*0.05 + 0.10*0.70 = 0.045 + 0.070 = 0.115

E_costo_dado_operativa = P_dañada_dado_operativa * costo_reparacion   # 250,000
E_costo_dado_dañada    = P_dañada_dado_dañada * costo_reparacion      # 3,500,000
E_costo_mañana = P_operativa_hoy*E_costo_dado_operativa + P_dañada_hoy*E_costo_dado_dañada
# = 0.90*250,000 + 0.10*3,500,000 = 225,000 + 350,000 = 575,000
```

**Resultados:** $P(\text{Dañada mañana}) = 0.115$ (11.5%) y $E[\text{costo mañana}] = \$575{,}000$ COP.

**Por qué importa esto:** nota que puedes llegar al mismo costo esperado por dos caminos equivalentes — (a) calculando primero $P(\text{Dañada mañana})=0.115$ y multiplicando por el costo fijo de reparación (\$5,000,000 × 0.115 = \$575,000), o (b) promediando los costos esperados condicionales. Que ambos caminos coincidan **no es casualidad**: es el mismo principio (la ley de valor esperado total) aplicado de dos formas. Esta dualidad —condicionar sobre "lo que puede pasar hoy" para predecir "mañana"— es exactamente el mecanismo que vas a usar en la Semana 2 para calcular probabilidades de varios pasos en una cadena de Markov, y en el Bloque B cuando calcules pronósticos condicionados a la información disponible hasta hoy.
</details>

## 2. ¿Qué es un proceso estocástico? — *(O2)*

> **Definición (Ross, cap. 2).** Un proceso estocástico es una colección de variables aleatorias $\{X(t), t \in T\}$, donde para cada $t$ en el conjunto de índices $T$, $X(t)$ es una variable aleatoria.

La diferencia clave con lo que ya conoces de Simulación: antes generabas *una* variable aleatoria (ej. el tiempo de servicio de un cliente). Ahora tienes una **familia completa** de variables aleatorias indexadas por el tiempo (o por algún otro parámetro).

### Actividad 2 — ¿Variable aleatoria o proceso estocástico? *(O2, formativa)*

Para cada escenario, decide si se describe **una variable aleatoria simple** o **un proceso estocástico**, y justifica en una frase.

**a)** El tiempo de servicio de un único cliente en un banco (lo que ya simulabas en Simulación).

**b)** El nivel de un embalse, medido una vez al mes, durante los próximos 5 años.

**c)** El tiempo de servicio de cada uno de los primeros 50 clientes de hoy, registrado en el orden en que llegaron.

In [3]:
# Escribe tu respuesta (variable aleatoria / proceso estocástico) y tu justificación
respuesta_a = "..."
respuesta_b = "..."
respuesta_c = "..."


<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

**a) Variable aleatoria simple.** Es un único número aleatorio — no hay una familia indexada por tiempo, solo un resultado.

**b) Proceso estocástico, con dependencia (memoria).** Hay 60 variables aleatorias indexadas por el mes $t=1,\ldots,60$, y —a diferencia del inciso (c)— el nivel del mes siguiente depende fuertemente del nivel actual (un embalse no se vacía ni se llena de la nada de un mes a otro). Este es el tipo de estructura que vamos a formalizar con cadenas de Markov.

**c) Proceso estocástico — pero un caso particular sutil.** Técnicamente sí es un proceso estocástico (una familia de 50 variables aleatorias indexadas por el orden de llegada), pero si los tiempos de servicio de distintos clientes no dependen entre sí, es un proceso de **variables aleatorias independientes**. Esto es útil que lo notes ya: *todo proceso con dependencia entre sus variables es un proceso estocástico, pero no todo proceso estocástico necesita tener dependencia*. La independencia es el caso "aburrido" — el curso se pone interesante quiando hay dependencia real, como en (b), porque ahí es donde el pasado sí te ayuda a predecir el futuro.

**Conexión con el Bloque B:** cuando en la Semana 9 hablemos de series de tiempo, vas a reconocer inmediatamente que un dato observado mes a mes (como en el inciso b) es justamente el tipo de proceso estocástico que después vamos a pronosticar.
</details>

## 3. Clasificación de procesos estocásticos — *(O3)*

Un proceso estocástico se clasifica según dos ejes independientes:

| | **Tiempo discreto** | **Tiempo continuo** |
|---|---|---|
| **Espacio de estados discreto** | Cadena de Markov discreta (Semana 2–4) | Cadena de Markov en tiempo continuo (Semana 6) |
| **Espacio de estados continuo** | Series de tiempo (Bloque B) | Movimiento browniano (mencionado, no profundizamos) |

### Actividad 3 — Clasificando procesos de tu propio campo *(O3, formativa)*

Para cada escenario, identifica: **(a)** el espacio de estados (discreto o continuo) y **(b)** el parámetro temporal (discreto o continuo).

1. El número de unidades defectuosas producidas cada hora en una línea de ensamble.
2. El nivel de inventario de un producto, medido en kilogramos, en cualquier instante del día.
3. El estado de una máquina (Operativa / En mantenimiento / Dañada), revisado cada turno.
4. El precio de una acción, observado continuamente durante la jornada bursátil.

In [4]:
# Escribe aquí tu clasificación para cada escenario antes de continuar
respuesta_1 = "..."  # espacio de estados: ?, tiempo: ?
respuesta_2 = "..."
respuesta_3 = "..."
respuesta_4 = "..."


<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

1. **Discreto / discreto.** El conteo de unidades defectuosas solo puede tomar valores enteros (0, 1, 2…) y se observa una vez por hora — un índice temporal discreto. Este es el tipo de proceso que vamos a formalizar como cadena de Markov en la Semana 2.

2. **Continuo / continuo.** El inventario en kilogramos puede tomar cualquier valor real, y "en cualquier instante" significa que el tiempo también es continuo. Este tipo de proceso lo dejamos fuera del alcance profundo del curso, pero es bueno que lo reconozcas — es el terreno de las ecuaciones diferenciales estocásticas.

3. **Discreto / discreto.** Tres estados posibles, revisados en momentos puntuales (por turno). Este es el ejemplo *más representativo* de lo que vamos a estudiar intensamente las próximas tres semanas — de hecho, es casi idéntico al ejemplo de la máquina que vimos en la Actividad 1.

4. **Continuo / continuo.** El precio es un número real y se observa (en principio) en todo instante. Cuando en el Bloque B trabajemos con series de tiempo, en realidad vamos a *discretizar* el tiempo (precios de cierre diarios, por ejemplo) — vale la pena que notes desde ya que forecasting es, estrictamente, un caso particular de proceso estocástico con muestreo discreto de un fenómeno continuo. Esa es la idea que vamos a retomar explícitamente en la clase puente (Semana 9).

**Por qué importa esta clasificación:** determina qué herramienta matemática te sirve. Los escenarios 1 y 3 son el tipo que Ross cubre con cadenas de Markov discretas — el corazón de las próximas tres semanas.
</details>

## 4. Tu primer proceso estocástico simulado: la caminata aleatoria — *(O4)*

Vamos a simular el ejemplo más simple posible de proceso estocástico de tiempo discreto y espacio discreto: una caminata aleatoria simple. En cada paso, el proceso sube 1 con probabilidad $p$ o baja 1 con probabilidad $1-p$.

> 🎓 **Esta función se construye en vivo, en clase, paso a paso.** Si no asististe, pídele a un compañero sus notas o revisa la grabación — el código completo lo necesitas para la Actividad 4.

### Actividad 4 — El efecto del sesgo $p$ *(O4, formativa)*

Hasta ahora usamos $p=0.5$ (caminata sin sesgo). En un proceso real de IE (Ingeniería Industrial), $p$ rara vez es exactamente 0.5 — por ejemplo, si $X(t)$ representa el inventario acumulado y hay más demanda que reposición, esperarías un sesgo hacia abajo ($p<0.5$).

**Tarea:** simula **2,000 trayectorias** de 200 pasos para cada uno de $p \in \{0.3,\ 0.5,\ 0.7\}$, guarda la posición final $X(200)$ de cada trayectoria, y calcula el promedio empírico de esas posiciones finales para cada $p$.

**Pista teórica:** en cada paso, $E[\text{paso}] = p\cdot(1) + (1-p)\cdot(-1) = 2p-1$. Por linealidad del valor esperado, $E[X(n)] = n\cdot(2p-1)$. Verifica si tu resultado empírico se acerca a esta fórmula.

In [6]:
# YOUR CODE HERE
resultados = {}
for p in [0.3, 0.5, 0.7]:
    finales = []  # TODO: lista/array con las 2000 posiciones finales X(200) para este p
    resultados[p] = np.nan  # TODO: reemplaza por np.mean(finales) una vez completes la lista

for p, media in resultados.items():
    print(f"p={p}: media empírica de X(200) = {media:.2f}  |  teórica = {200*(2*p-1):.2f}")

p=0.3: media empírica de X(200) = nan  |  teórica = -80.00
p=0.5: media empírica de X(200) = nan  |  teórica = 0.00
p=0.7: media empírica de X(200) = nan  |  teórica = 80.00


<details>
<summary><b>▶ Revelar solución y explicación</b></summary>

```python
resultados = {}
for p in [0.3, 0.5, 0.7]:
    finales = [caminata_aleatoria(200, p, semilla=1000 + i)[-1] for i in range(2000)]
    resultados[p] = np.mean(finales)

for p, media in resultados.items():
    print(f"p={p}: media empírica de X(200) = {media:.2f}  |  teórica = {200*(2*p-1):.2f}")
```

**Resultado esperado (aprox.):** p=0.3 → media ≈ −80; p=0.5 → media ≈ 0; p=0.7 → media ≈ +80 — muy cerca de la fórmula teórica $200\cdot(2p-1)$ en los tres casos.

**Por qué importa esto:** acabas de verificar empíricamente, con tu propia simulación, que el valor esperado de un proceso estocástico en el instante $n$ se puede predecir *sin simular nada* si conoces la estructura del proceso — solo necesitas la fórmula $E[X(n)] = n(2p-1)$. Esta es la primera muestra de algo que vas a explotar constantemente el resto del curso: cuando el proceso tiene una estructura conocida (Markov, Poisson, etc.), casi nunca necesitas simular miles de trayectorias para responder preguntas de largo plazo — puedes calcular la respuesta analíticamente. La simulación te sirve para *verificar* que tu álgebra está bien, no para reemplazarla. Esa es exactamente la lógica que vas a usar en el Taller 01 (Semana 2) al comparar tu distribución estacionaria simulada contra el cálculo analítico.
</details>

## 5. Autoevaluación rápida de cierre

Antes de pasar a la Semana 2, verifica que puedes responder esto sin mirar atrás (son conceptuales, no de cálculo):

1. ¿Por qué el tiempo de servicio de un solo cliente **no** es un proceso estocástico, pero el nivel mensual de un embalse durante 5 años sí lo es?
2. Si dos procesos tienen exactamente el mismo espacio de estados y el mismo parámetro temporal, ¿son necesariamente el mismo proceso? (Pista: piensa en las probabilidades de transición del ejemplo de la máquina vs. otra máquina con distinta tasa de daño.)
3. En la Actividad 4, ¿por qué el promedio de 2000 trayectorias se parece tanto a la fórmula teórica, y qué pasaría si solo hubieras simulado 5 trayectorias en vez de 2000?

<details>
<summary><b>▶ Revelar respuestas</b></summary>

1. Una variable aleatoria simple es un único número; un proceso estocástico es una *familia* de variables aleatorias indexadas (por tiempo, en nuestro caso). El embalse tiene 60 variables (una por mes) — es una familia. El cliente único es un solo resultado.

2. No. El espacio de estados y el parámetro temporal describen la *forma* del proceso, pero no sus probabilidades. Dos cadenas de Markov con los mismos 3 estados y tiempo discreto pueden comportarse de manera completamente distinta si sus matrices de transición son diferentes — igual que dos personas pueden tener la misma edad y nacionalidad pero personalidades opuestas.

3. Con 2000 trayectorias, el promedio empírico converge al valor esperado teórico por la Ley de los Grandes Números (que ya viste en Simulación). Con solo 5 trayectorias, el promedio podría alejarse bastante del valor teórico por puro azar — la varianza del promedio muestral es mucho más alta con pocas muestras. Este es el mismo principio que ya aplicabas al decidir cuántas réplicas correr en un modelo de Simulación.
</details>

## 6. Cierre y puente a la próxima semana

Hoy formalizamos qué es un proceso estocástico y por qué la probabilidad condicional es la herramienta que lo describe. La próxima semana (S2) vamos a estudiar el caso más importante para este curso — y el que vas a usar en tu **primer taller calificado**: las **cadenas de Markov discretas**, donde la probabilidad condicional de "a dónde voy mañana" depende *solo* de "dónde estoy hoy", no de todo tu historial. Esa propiedad tiene nombre (propiedad de Markov) y consecuencias matemáticas muy poderosas — entre ellas, la matriz de transición que vas a construir y simular en el taller.

**Antes de la próxima clase:** repasa multiplicación de matrices — la vas a necesitar desde el primer minuto.

---
**Referencias de esta semana:** Ross, S. M. *Introduction to Probability Models*, cap. 1, sec. 1.4 (probabilidad condicional); cap. 2, sec. 2.4 (valor esperado) y sec. 2.9 (introducción a procesos estocásticos); cap. 3 (valor esperado condicional y cálculo de probabilidades por condicionamiento).